In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/fake-news-detection/true.csv
/kaggle/input/fake-news-detection/fake.csv


## Setting Up the Environment

In [1]:
# !pip install torch torch-geometric spektral
# !pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+${CUDA}.html

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import networkx as nx
from spektral.utils import normalized_adjacency

## Data Loading and Preprocessing

In [4]:
# Load datasets
fake_df = pd.read_csv('/kaggle/input/fake-news-detection/fake.csv')
true_df = pd.read_csv('/kaggle/input/fake-news-detection/true.csv')

# Add labels
fake_df['label'] = 1  # 1 for fake
true_df['label'] = 0  # 0 for true

In [5]:
print(fake_df.shape)
print(true_df.shape)

# Combine datasets
df = pd.concat([fake_df, true_df], ignore_index=True)

(23481, 5)
(21417, 5)


In [6]:
import string

def preprocess_text(text):
    # Convert the text to lowercase
    text = text.lower()
    # Remove all punctuation marks
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove extra whitespace (leading, trailing, and multiple spaces)
    text = ' '.join(text.split())
    return text

# Apply text preprocessing to the 'text' and 'title' columns
df['text'] = df['text'].astype(str).apply(preprocess_text)
df['title'] = df['title'].astype(str).apply(preprocess_text)

# Combine the 'title' and 'text' columns into a single 'content' column
df['content'] = df['title'] + " " + df['text']

# Sample a subset of the data to manage computational resources
sample_size = 6000
df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

# Print the number of sampled articles
print(f"Sampled {sample_size} articles for analysis.")

Sampled 6000 articles for analysis.


In [7]:
# Extract the 'label' column as the target variable (y_data)
y_data = df_sample['label'].values  

# Drop the 'label' column from the DataFrame to avoid leakage during model training
df_sample.drop(['label'], axis=1, inplace=True)

# Display the extracted target variable
y_data

array([1, 0, 0, ..., 1, 1, 0])

In [8]:
df_sample.head(10)

,title,text,subject,date,content
0,ben stein calls out 9th circuit court committe...,21st century wire says ben stein reputable pro...,US_News,"February 13, 2017",ben stein calls out 9th circuit court committe...
1,trump drops steve bannon from national securit...,washington reuters us president donald trump r...,politicsNews,"April 5, 2017",trump drops steve bannon from national securit...
2,puerto rico expects us to lift jones act shipp...,reuters puerto rico governor ricardo rossello ...,politicsNews,"September 27, 2017",puerto rico expects us to lift jones act shipp...
3,oops trump just accidentally confirmed he leak...,on monday donald trump once again embarrassed ...,News,"May 22, 2017",oops trump just accidentally confirmed he leak...
4,donald trump heads for scotland to reopen a go...,glasgow scotland reuters most us presidential ...,politicsNews,"June 24, 2016",donald trump heads for scotland to reopen a go...
5,paul ryan responds to dem’s sitin on gun contr...,on wednesday democrats took a powerful stance ...,News,"June 22, 2016",paul ryan responds to dem’s sitin on gun contr...
6,awesome diamond and silk rip into the press “w...,president trump s rally in fl on saturday was ...,Government News,"Feb 19, 2017",awesome diamond and silk rip into the press “w...
7,stand up and cheer ukip party leader slams ger...,he s been europe s version of the outspoken te...,left-news,"Mar 8, 2016",stand up and cheer ukip party leader slams ger...
8,north korea shows no sign it is serious about ...,washington reuters the state department said w...,worldnews,"December 13, 2017",north korea shows no sign it is serious about ...
9,trump signals willingness to raise us minimum ...,this version of the story corrects the figure ...,politicsNews,"May 4, 2016",trump signals willingness to raise us minimum ...


## Feature Extraction

In [9]:
# Initialize the TF-IDF vectorizer with a limit of 2000 features
vectorizer = TfidfVectorizer(max_features=2000)  
# Transform the 'content' column into a TF-IDF feature matrix
tfidf_features = vectorizer.fit_transform(df_sample['content']).toarray()

from sklearn.preprocessing import StandardScaler

# Normalize the TF-IDF features using StandardScaler
scaler = StandardScaler()
tfidf_features = scaler.fit_transform(tfidf_features)

# Print the shape of the resulting TF-IDF feature matrix
print(f"TF-IDF feature matrix shape: {tfidf_features.shape}")

TF-IDF feature matrix shape: (6000, 2000)


## Constructing the Graph

In [10]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

k = 5  # Number of nearest neighbors

# Initialize NearestNeighbors with cosine similarity as the distance metric
nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto', metric='cosine').fit(tfidf_features)

# Find the k+1 nearest neighbors (including the point itself)
distances, indices = nbrs.kneighbors(tfidf_features)

# Initialize lists to store the rows and columns of the adjacency matrix
rows = []
cols = []

# Build the adjacency matrix by adding edges between the nodes and their neighbors
for i in range(tfidf_features.shape[0]):
    for j in range(1, k+1):  # Skip the first neighbor (itself)
        neighbor = indices[i][j]
        rows.append(i)
        cols.append(neighbor)

# Create a sparse adjacency matrix where each edge has a value of 1
data = np.ones(len(rows))
adjacency_sparse = csr_matrix((data, (rows, cols)), shape=(tfidf_features.shape[0], tfidf_features.shape[0]))

# Symmetrize the adjacency matrix to ensure the graph is undirected (bidirectional edges)
adjacency_sparse = adjacency_sparse.maximum(adjacency_sparse.transpose())

# Print the number of nodes and edges in the sparse adjacency matrix
print(f"Number of nodes: {adjacency_sparse.shape[0]}")
print(f"Number of edges: {adjacency_sparse.count_nonzero()}")

Number of nodes: 6000
Number of edges: 45085


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

export_dir = Path("feature_extraction") # Create an output folder after feature extraction
export_dir.mkdir(parents=True, exist_ok=True)

# Save node information
nodes_df = pd.DataFrame({
    "node_id": np.arange(df_sample.shape[0]),
    "label": y_data
})
if "title" in df_sample.columns:
    nodes_df["title"] = df_sample["title"].values
nodes_df.to_csv(export_dir / "nodes.csv", index=False)

# Convert sparse adjacency to COO to export edge list and edge_index
adjacency_coo = adjacency_sparse.tocoo()

# Save edges for visualization (unique undirected edges)
edges_df = pd.DataFrame({
    "source": adjacency_coo.row,
    "target": adjacency_coo.col,
    "weight": adjacency_coo.data
})
edges_df = edges_df[edges_df["source"] < edges_df["target"]].reset_index(drop=True)
edges_df.to_csv(export_dir / "edges.csv", index=False)

# Save edge_index in [2, num_edges] format for graph libraries
edge_index = np.vstack((adjacency_coo.row, adjacency_coo.col)).astype(np.int64)
np.save(export_dir / "edge_index.npy", edge_index)

print(f"Saved nodes to: {export_dir / 'nodes.csv'}")
print(f"Saved edges to: {export_dir / 'edges.csv'}")
print(f"Saved edge_index to: {export_dir / 'edge_index.npy'}")